# Notebook 01 — Exploração: Complexidade de Atenção e Sinais de Áudio

**Aula 02 — Desafios de Performance em NLP e Áudio**  
Disciplina: Engenharia de Machine Learning — FIAP PósTech

---

## Objetivos

1. Explorar o dataset sintético de configurações de atenção e áudio.
2. Visualizar a complexidade quadrática O(n²) do self-attention vanilla (Vaswani et al., 2017).
3. Gerar e analisar sinais de áudio sintéticos com STFT e espectrogramas Mel.
4. Identificar relações entre parâmetros (seq_length, d_model, num_heads) e latência/memória.

## Fundamentação Teórica

Conforme discutido na seção *Fundamentos do Mecanismo de Self-Attention*:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

A matriz de atenção $QK^T$ é $n \times n$, implicando **complexidade de memória O(n²)**.
Para modelos como GPT-3 com 96 camadas e múltiplas cabeças, os requisitos escalam
rapidamente para dezenas de gigabytes (Brown et al., 2020).

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adicionar src/ ao path
sys.path.insert(0, os.path.abspath(".."))

from src.data_preprocessing import (
    DataPreprocessor,
    generate_synthetic_audio,
    compute_stft,
    compute_mel_spectrogram,
    compute_mfcc,
    pad_sequences,
    sorted_batch_indices,
)
from src.utils import format_bytes

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Imports carregados com sucesso.")

## 1. Carga e Exploração do Dataset

O dataset contém metadados de configurações de atenção e parâmetros
de áudio, projetado para demonstrar os conceitos da seção *Saiba Mais*.

In [ ]:
# Gerar dataset se não existir, depois carregar
preprocessor = DataPreprocessor(seed=42)
dataset_path = os.path.join("..", "data", "raw", "dataset.csv")

if not os.path.exists(dataset_path):
    print("Gerando dataset sintético...")
    preprocessor.generate_dataset(output_path=dataset_path)

df = preprocessor.load_data(dataset_path)
print(f"Dataset: {df.shape[0]} amostras, {df.shape[1]} colunas")
df.head(10)

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Distribuição de categorias e tipos de atenção
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["category"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "coral", "seagreen"])
axes[0].set_title("Distribuição por Categoria de Sequência")
axes[0].set_ylabel("Contagem")

df["attention_type"].value_counts().plot.bar(ax=axes[1], color="mediumpurple")
axes[1].set_title("Distribuição por Tipo de Atenção")
axes[1].set_ylabel("Contagem")

plt.tight_layout()
plt.show()

## 2. Complexidade Quadrática do Self-Attention

Conforme a seção *Fundamentos do Mecanismo de Self-Attention*:

> A matriz de atenção $QK^T$ possui dimensões $n \times n$. Isso implica
> complexidade de tempo O(n²·d) e, crucialmente, **complexidade de memória O(n²)**
> para armazenar os scores de atenção.

Vamos visualizar isso empiricamente no dataset.

In [ ]:
# Latência por seq_length e tipo de atenção
# Evidencia o crescimento O(n²) para vanilla vs O(n) para linear
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Latência mediana por grupo
latency_pivot = df.groupby(["seq_length", "attention_type"])["vanilla_latency_ms"].median().reset_index()
sns.lineplot(data=latency_pivot, x="seq_length", y="vanilla_latency_ms",
             hue="attention_type", marker="o", ax=axes[0])
axes[0].set_title("Latência Mediana vs Comprimento de Sequência\n(Escala log — O(n²) vs O(n))")
axes[0].set_xlabel("Comprimento da Sequência (n)")
axes[0].set_ylabel("Latência (ms)")
axes[0].set_yscale("log")

# Memória por grupo
mem_pivot = df.groupby(["seq_length", "attention_type"])["memory_mb"].median().reset_index()
sns.lineplot(data=mem_pivot, x="seq_length", y="memory_mb",
             hue="attention_type", marker="s", ax=axes[1])
axes[1].set_title("Memória Mediana vs Comprimento de Sequência")
axes[1].set_xlabel("Comprimento da Sequência (n)")
axes[1].set_ylabel("Memória (MB)")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# Memória teórica da matriz de atenção n×n
# Conforme seção 'Fundamentos': para FP16, seq=2048, uma camada = 8 MB
from src.evaluation import attention_memory_theoretical

seq_lengths = [64, 128, 256, 512, 1024, 2048, 4096]
types = ["vanilla", "linear", "flash", "sparse", "local"]

print(f"{'Seq Len':>8} | " + " | ".join(f"{t:>10}" for t in types))
print("-" * 75)
for n in seq_lengths:
    row = f"{n:>8} | "
    row += " | ".join(
        f"{format_bytes(attention_memory_theoretical(n, 128, 4, t)):>10}"
        for t in types
    )
    print(row)

## 3. Correlações entre Parâmetros

A seção *Decomposição da Latência em Transformers* destaca que
otimizações focadas apenas em redução de FLOPs podem ter **impacto
limitado** se o gargalo real for acesso à memória.

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlação entre Parâmetros de Configuração e Métricas")
plt.tight_layout()
plt.show()

## 4. Exploração de Sinais de Áudio

Conforme a seção *Processamento Digital de Sinais para Áudio*:

> O Teorema de Nyquist-Shannon estabelece que para reconstrução perfeita
> de um sinal, a taxa de amostragem deve ser pelo menos o dobro da
> frequência máxima presente. Para fala humana (fundamental até ~4 kHz),
> taxas de 16 kHz são suficientes para ASR.

### 4.1 Geração e Visualização de Sinais Sintéticos

In [ ]:
# Gerar sinais de áudio sintéticos com diferentes frequências
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

for i, (freq, label) in enumerate([(220, "220 Hz (Nota A3)"),
                                     (440, "440 Hz (Nota A4)"),
                                     (880, "880 Hz (Nota A5)")]):
    signal, sr = generate_synthetic_audio(freq_hz=freq, duration_s=0.05, sample_rate=16000)
    t = np.arange(len(signal)) / sr * 1000  # ms
    axes[i].plot(t, signal, color="steelblue", linewidth=0.8)
    axes[i].set_title(f"Sinal: {label} @ {sr} Hz")
    axes[i].set_ylabel("Amplitude")

axes[-1].set_xlabel("Tempo (ms)")
plt.tight_layout()
plt.show()

In [ ]:
# Efeito do ruído no sinal (noise_level do dataset)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, noise in enumerate([0.0, 0.1, 0.4]):
    signal, sr = generate_synthetic_audio(
        freq_hz=440, duration_s=0.05, sample_rate=16000, noise_level=noise
    )
    t = np.arange(len(signal)) / sr * 1000
    axes[i].plot(t, signal, color="coral", linewidth=0.8)
    axes[i].set_title(f"Ruído = {noise}")
    axes[i].set_xlabel("Tempo (ms)")
    axes[i].set_ylabel("Amplitude")

plt.suptitle("Impacto do Nível de Ruído no Sinal de 440 Hz", fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 STFT e Espectrograma Mel

A STFT (Short-Time Fourier Transform) é a ferramenta fundamental para
análise tempo-frequência:

$$X[m, k] = \sum_{n=0}^{N-1} x[n + mH] \cdot w[n] \cdot e^{-j2\pi kn/N}$$

Os parâmetros `n_fft=400` e `hop_length=160` correspondem a janelas
de **25 ms** com stride de **10 ms** para áudio a 16 kHz — configuração
padrão do Whisper (Radford et al., 2023).

In [ ]:
# Gerar sinal mais longo para STFT
signal, sr = generate_synthetic_audio(
    freq_hz=440, duration_s=1.0, sample_rate=16000, noise_level=0.05
)
print(f"Sinal: {len(signal)} amostras, {len(signal)/sr:.2f}s, sr={sr} Hz")

# STFT com parâmetros padrão do Whisper
stft_mag = compute_stft(signal, n_fft=400, hop_length=160)
print(f"STFT shape: {stft_mag.shape} (n_freq_bins × n_frames)")
print(f"  → {stft_mag.shape[0]} bins de frequência")
print(f"  → {stft_mag.shape[1]} frames temporais")
print(f"  → Resolução temporal: {160/sr*1000:.1f} ms/frame")
print(f"  → Resolução frequencial: {sr/400:.1f} Hz/bin")

# Visualizar STFT
fig, ax = plt.subplots(figsize=(12, 4))
duration = stft_mag.shape[1] * 160 / sr
img = ax.imshow(
    20 * np.log10(stft_mag + 1e-10),
    aspect="auto", origin="lower",
    extent=[0, duration, 0, sr/2],
    cmap="magma"
)
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Frequência (Hz)")
ax.set_title("Espectrograma (STFT) — n_fft=400, hop=160 (configuração Whisper)")
plt.colorbar(img, ax=ax, label="dB")
plt.tight_layout()
plt.show()

In [ ]:
# Espectrograma Mel — escala perceptual
# m = 2595 · log10(1 + f/700)
mel_db = compute_mel_spectrogram(
    signal, sample_rate=sr, n_mels=80, n_fft=400, hop_length=160
)
print(f"Mel spectrogram shape: {mel_db.shape} (n_mels × n_frames)")

fig, ax = plt.subplots(figsize=(12, 4))
n_frames = mel_db.shape[1]
duration = n_frames * 160 / sr
img = ax.imshow(
    mel_db, aspect="auto", origin="lower",
    extent=[0, duration, 0, 80],
    cmap="magma"
)
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Filtro Mel")
ax.set_title("Espectrograma Mel (80 filtros) — Escala perceptual\n"
             "m = 2595 · log₁₀(1 + f/700) (Radford et al., 2023)")
plt.colorbar(img, ax=ax, label="dB")
plt.tight_layout()
plt.show()

In [ ]:
# MFCCs — representação compacta
mfccs = compute_mfcc(signal, sample_rate=sr, n_mfcc=13, n_mels=80)
print(f"MFCC shape: {mfccs.shape} (n_mfcc × n_frames)")

fig, ax = plt.subplots(figsize=(12, 4))
img = ax.imshow(mfccs, aspect="auto", origin="lower", cmap="coolwarm")
ax.set_xlabel("Frame")
ax.set_ylabel("Coeficiente MFCC")
ax.set_title("MFCCs (13 coeficientes) — DCT sobre log Mel spectrogram")
plt.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

## 5. Estratégias de Batching

Conforme a seção *Estratégias de Batching para Sequências Variáveis*:

> O desperdício computacional [do padding] é proporcional à variância
> dos comprimentos: batches com uma sequência muito longa e muitas
> curtas desperdiçam até 90% dos FLOPs em processamento de padding.

A técnica de **sorted batching** (Ott et al., 2019) agrupa sequências
de comprimentos similares para minimizar padding.

In [ ]:
# Demonstração: padding waste com e sem sorted batching
rng = np.random.default_rng(42)
lengths = rng.choice([16, 32, 64, 128, 256, 512], size=100).tolist()

batch_size = 8

# Batches aleatórios
random_batches = [lengths[i:i+batch_size] for i in range(0, len(lengths), batch_size)]
random_waste = sum(max(b) * len(b) - sum(b) for b in random_batches)
random_total = sum(max(b) * len(b) for b in random_batches)

# Batches ordenados (sorted batching)
sorted_idx = sorted_batch_indices(lengths, batch_size)
sorted_batches = [[lengths[i] for i in batch] for batch in sorted_idx]
sorted_waste = sum(max(b) * len(b) - sum(b) for b in sorted_batches)
sorted_total = sum(max(b) * len(b) for b in sorted_batches)

print("=== Comparação de Estratégias de Batching ===")
print(f"Random: {random_waste:,} tokens desperdiçados ({100*random_waste/random_total:.1f}%)")
print(f"Sorted: {sorted_waste:,} tokens desperdiçados ({100*sorted_waste/sorted_total:.1f}%)")
print(f"Redução de desperdício: {100*(1 - sorted_waste/random_waste):.1f}%")

In [ ]:
# Demonstração de pad_sequences com attention mask
sequences = [
    np.array([1, 2, 3]),
    np.array([4, 5, 6, 7, 8]),
    np.array([9, 10]),
]
padded, mask = pad_sequences(sequences)
print("Sequências originais:", [s.tolist() for s in sequences])
print(f"\nPadded (shape {padded.shape}):")
print(padded)
print(f"\nAttention mask (shape {mask.shape}):")
print(mask)
print("\n→ Posições com mask=0 são IGNORADAS no cálculo de softmax")

## Resumo

Neste notebook exploramos:

1. **Dataset**: 5000 amostras com configurações de atenção e áudio.
2. **Complexidade O(n²)**: Crescimento quadrático de latência e memória para self-attention vanilla.
3. **Processamento de áudio**: STFT, Mel spectrograms e MFCCs com parâmetros padrão do Whisper.
4. **Batching**: Sorted batching reduz significativamente o desperdício de padding.

---

**Próximo notebook:** [02_treinamento.ipynb](02_treinamento.ipynb) — Benchmark de mecanismos de atenção e treinamento de modelos.